In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

In [ ]:
"""
2-3 Tree Implementation:

A 2-3 tree is a balanced search tree in which:
- 2-nodes have one key and two children.
- 3-nodes have two keys and three children.
- All the leaf nodes lie at the same depth.

The tree is balanced by splitting overfull nodes (temporary nodes with 3 keys
and 4 children) into two 2-nodes, and passing the middle key up the tree.

The tree depth can only be increased by a split operation at the root node: this
is how the tree maintains balance. Splitting is a constant time operation and
only acts locally in the structure.

Deletion works using merge and redistribute tree operations. 

Searching a 2-3 tree of N nodes is an O(logN) operation; inserting a node into
a 2-3 tree of size N is also an O(logN) operation. The depth of the tree ranges
from log N / log 2 (log base 2 of N), to log N / log 3 (log base 3 of N).
"""


class TwoThreeNode:
    """A node in a 2-3 tree. Stores keys and children as lists"""

    def __init__(self, keys=None, children=None):
        self.keys = keys or []
        self.children = children or []

    def is_leaf(self) -> bool:
        return len(self.children) == 0


class TwoThreeTree(AbstractSearchInterface):
    """A 2-3 balanced search tree with search, insert and delete operations."""

    def __init__(self, root=TwoThreeNode | None):
        self.root = root

    def __get_child_index(self, element: str, node: TwoThreeNode) -> int:
        """Returns the index of the child (in node.children) to descend the
        tree to based on a given element.

        The index of the first key greater than the element is the index of the
        correct child. (If no key is greater, then descend to the rightmost
        child)
        """
        child_index = len(node.keys)
        for i, key in enumerate(node.keys):
            if element < key:
                child_index = i
                break
        return child_index

    # --- Search ---

    def __search(self, element: str, node: TwoThreeNode) -> bool:
        """Recursively searches the tree rooted at node for element. Returns
        true if found.
        """
        if node is None:
            return False
        if element in node.keys:
            return True
        # Leaf reached without finding the element — the element is not in the
        # tree
        if node.is_leaf():
            return False
        # Get the index of the child to descend to
        child_index = self.__get_child_index(element, node)
        return self.__search(element, node.children[child_index])

    def searchElement(self, element: str) -> bool:
        """Returns true if and only if the 2-3 tree contains a specified
        element. Delegates work to a recursive helper, search.
        """
        if self.root is None:
            return False
        return self.__search(element, self.root)

    # --- Insert ---

    def __insert(
        self, element: str, node: TwoThreeNode
    ) -> None | tuple[str, TwoThreeNode, TwoThreeNode]:
        """Insert an element into a 2-3 subtree rooted at a given node, keeping
        the balance in check. This is a recursive helper method for insert.

        Returns:
          None : if the insertion is absorbed by the node without causing a
                 split operation
          middle_key, left_child, right_child (_, Node, Node) : if the node
                 split, the calling method must absorb the key being passed
                 up, and the left and right children
        """
        # Base case — leaf node
        if node.is_leaf():
            node.keys.append(element)
            node.keys.sort()

            # Still a valid 2- or 3-node
            if len(node.keys) <= 2:
                return None

            # Overfull (has 3 keys) — split and pass the middle key up the
            # call chain
            middle_key = node.keys[1]
            left_child = TwoThreeNode([node.keys[0]])
            right_child = TwoThreeNode([node.keys[2]])
            return (middle_key, left_child, right_child)

        # Recursive case — 2- or 3- (non-leaf) node
        # Find the correct child to recurse to
        child_index = self.__get_child_index(element, node)
        result = self.__insert(element, node.children[child_index])

        # Child absorbed the insert without splitting — so do nothing
        if result is None:
            return None

        # A child split, so absorb the received key and new children
        middle_key, left_child, right_child = result
        node.keys.append(middle_key)
        node.keys.sort()

        # Replace the child at child_index with the two new children
        node.children = (
            node.children[:child_index]
            + [left_child, right_child]
            + node.children[child_index + 1 :]
        )

        # Still a valid 2- or 3-node — so do not initiate a split
        if len(node.keys) <= 2:
            return None

        # Overfull (has 3 keys) — split and pass (as before) but attach
        # children. Left child node gets the first two children, right child
        # node gets the last two
        middle_key = node.keys[1]
        left_child = TwoThreeNode([node.keys[0]], node.children[:2])
        right_child = TwoThreeNode([node.keys[2]], node.children[2:])
        return (middle_key, left_child, right_child)

    def insertElement(self, element: str) -> bool:
        """Insert an element into a 2-3 tree, keeping the balance in check.
        Delegates work to the recursive __insert method.
        """
        # Return False if node already in the tree
        if self.__search(element, self.root):
            return False

        # If the tree is empty, then create a single leaf root
        if self.root is None:
            self.root = TwoThreeNode([element])
            return True

        result = self.__insert(element, self.root)

        # If the root split, then create a new one, one level higher
        # This is the only way the tree can grow taller
        if result is not None:
            middle_key, left_child, right_child = result
            self.root = TwoThreeNode([middle_key], [left_child, right_child])

        return True

    # --- Delete ---

    def __get_minimum_key(self, node: TwoThreeNode) -> str:
        """Returns the value of the smallest key in a subtree rooted at node."""
        while not node.is_leaf():
            node = node.children[0]
        return node.keys[0]

    def __delete(self, element: TwoThreeNode, node: str) -> bool:
        """Deletes an element from a 2-3 subtree rooted at node, using
        recursion.

        Returns:
          bool : True if the deletion caused the node to become underfull;
                    False otherwise.
        """
        if element in node.keys:
            index = node.keys.index(element)

            # Case 1 (leaf): Node contains element and node is a leaf.
            if node.is_leaf():
                node.keys.pop(index)
                return len(node.keys) == 0

            # Case 2 (internal): Node contains element and node is not a leaf.
            else:
                # Swap the node with its in-order successor, and recurse into
                # the correct subtree
                node.keys[index] = self.__get_minimum_key(node.children[index + 1])
                underfull = self.__delete(node.keys[index], node.children[index + 1])

                # If recursive delete calls empty the child node, then fix it.
                if underfull:
                    self.__fix_underfull(node, index + 1)

                # True if node is now underfull
                return len(node.keys) == 0
        else:
            # Case 3 (leaf, not found): Element not in node and node is a leaf
            # — element is not in the tree.
            if node.is_leaf():
                return False

            # Case 4 (internal, not found): Element not in node, and node is
            # not a leaf. Recurse into the correct child node.
            child_index = self.__get_child_index(element, node)
            underfull = self.__delete(element, node.children[child_index])

            if underfull:
                self.__fix_underfull(node, child_index)

            # True if node is now underfull
            return len(node.keys) == 0

    def __fix_underfull(self, node: TwoThreeNode, child_index: int):
        """Fix a child node that has become empty, using redistribute and
        merge operations.
        """
        empty_child = node.children[child_index]

        # If the empty child has a left sibling
        if child_index > 0:
            left_sibling = node.children[child_index - 1]

            # Redistribute from left because left sibling is a 3-node
            if len(left_sibling.keys) == 2:
                empty_child.keys.insert(0, node.keys[child_index - 1])
                node.keys[child_index - 1] = left_sibling.keys.pop()
                if not left_sibling.is_leaf():
                    empty_child.children.insert(0, left_sibling.children.pop())

            # Merge left because left sibling is a 2-node
            else:
                left_sibling.keys.append(node.keys.pop(child_index - 1))
                left_sibling.children += empty_child.children
                node.children.pop(child_index)
            return

        # If the empty child has a right sibling (and did not have a left one)
        if child_index < len(node.children) - 1:
            right_sibling = node.children[child_index + 1]

            # Redistribute from right because right sibling is a 3-node
            if len(right_sibling.keys) == 2:
                empty_child.keys.append(node.keys[child_index])
                node.keys[child_index] = right_sibling.keys.pop(0)
                if not right_sibling.is_leaf():
                    empty_child.children.append(right_sibling.children.pop(0))

            # Merge right because right sibling is a 2-node
            else:
                right_sibling.keys.insert(0, node.keys.pop(child_index))
                right_sibling.children = empty_child.children + right_sibling.children
                node.children.pop(child_index)
            return

    def deleteElement(self, element: str) -> bool:
        """Deletes an element from a 2-3 tree, keeping the balance in check.
        Delegates work to recursive __delete method.

        Returns True if the element is removed from the list successfully;
        False if it was not in the tree to begin with.
        """
        if self.root is None:
            return False
        if not self.__search(element, self.root):
            return False

        underfull = self.__delete(element, self.root)

        # If deletion empties the root
        if underfull:
            if self.root.is_leaf():
                self.root = None
            else:
                self.root = self.root.children[0]
        return True

In [ ]:
'''Extremely basic test
'''

from random import shuffle 
bst = TwoThreeTree()


shuffled_even_integers = [i for i in range(0, 100, 2)]
shuffle(shuffled_even_integers)

for i in shuffled_even_integers:
  bst.insert(i)

positives = [] # Expect all even integers 0 - 100
negatives = [] # Expect all odd integers 0 - 100
integer_range = [i for i in range(100)]
for i in integer_range:
  if bst.search(i):
    positives += [i]
  else:
    negatives += [i]

print("The positives were: ", positives)
print("The negatives were: ", negatives)

# Note: I wrote this to verify my implementation works - we need to agree on a rigorous testing framework

The positives were:  [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98]
The negatives were:  [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 51, 53, 55, 57, 59, 61, 63, 65, 67, 69, 71, 73, 75, 77, 79, 81, 83, 85, 87, 89, 91, 93, 95, 97, 99]
